# TN10 trên Google Colab — "học tới nơi"

Notebook này lấy code từ **GitHub** và lưu dữ liệu, mô hình, kết quả, log trên **Google Drive**
(`MyDrive/PBL4_TN10`). Khi Colab ngắt kết nối thì mở lại notebook và chọn **Runtime → Run all**:
bước nào đã xong sẽ tự bỏ qua.

**Chuẩn bị (một lần):**
1. Push code lên GitHub, nhánh `mang-mo-rong-v3`.
2. *(Nên làm, tiết kiệm nhiều giờ sinh dữ liệu.)* Trên máy đang có dữ liệu, chạy `python -m scripts.dong_goi_tn10`,
   rồi tải `TN10_du_lieu.zip` (~0,7 GB) lên Drive vào thư mục `PBL4_TN10`.
3. Nếu repo để private: vào **Secrets** (biểu tượng chìa khoá bên trái), thêm `GITHUB_TOKEN` và bật quyền cho notebook.

**Runtime** (Runtime → Change runtime type):
- **GPU (T4 hoặc tốt hơn):** huấn luyện chạy trên GPU; lỗi GPU thì tự quay về CPU.
- Sinh dữ liệu và chấm luôn chạy trên CPU. Bản miễn phí chỉ có 2 CPU nên cần khoảng 2–3 phiên. Colab Pro chọn
  **High-RAM** để có nhiều CPU hơn.
- Để tab mở trong lúc chạy.

Tiêu chí chốt trước: `results/bao_da_tuyen/TIEU_CHI_10.md`. Hướng dẫn: `tai_lieu/HUONG_DAN_TN10.md`.

In [ ]:
# ===== Cấu hình =====
REPO = "https://github.com/huyth1204/PBL4.git"
NHANH = "mang-mo-rong-v3"
DRIVE = "/content/drive/MyDrive/PBL4_TN10"      # dữ liệu, mô hình, kết quả, log lưu ở đây
MA = "/content/PBL4"                            # mã nguồn, clone lại mỗi phiên
DAY_KET_QUA_LEN_GITHUB = False                  # True: đẩy kết quả lên nhánh tn10-colab (cần GITHUB_TOKEN có quyền ghi)

In [ ]:
# ===== 1. Gắn Google Drive =====
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
print("Drive:", DRIVE)

In [ ]:
# ===== 2. Lấy code từ GitHub =====
import subprocess


def _token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


TOKEN = _token()
URL = REPO.replace("https://", f"https://x-access-token:{TOKEN}@") if TOKEN else REPO


def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    if TOKEN:
        out = out.replace(TOKEN, "***")
    if r.returncode:
        raise SystemExit(f"git {a[0]} lỗi:\n{out}")
    return out


if os.path.isdir(f"{MA}/.git"):
    git("remote", "set-url", "origin", URL, cwd=MA)
    git("fetch", "origin", NHANH, cwd=MA)
    git("checkout", NHANH, cwd=MA)
    git("reset", "--hard", f"origin/{NHANH}", cwd=MA)      # bản clone tạm của Colab; dữ liệu nằm trên Drive
else:
    git("clone", "--branch", NHANH, "--single-branch", URL, MA)
print("Code:", git("log", "-1", "--format=%h  %ad  %s", "--date=iso", cwd=MA))

In [ ]:
# ===== 3. Thư viện + liên kết thư mục lưu trên Drive =====
import shutil
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "skyfield", "sgp4", "jplephem"], check=True)
os.chdir(MA)
sys.path.insert(0, MA)

LIEN_KET = ["data/bao_da_tuyen", "results/bao_da_tuyen/mo_hinh10", "results/bao_da_tuyen/tn10", "logs"]
for rel in LIEN_KET:
    goc, dich = os.path.join(MA, rel), os.path.join(DRIVE, rel)
    os.makedirs(dich, exist_ok=True)
    if os.path.islink(goc):
        continue
    if os.path.isdir(goc):          # thư mục có trong git: chép sang Drive (không ghi đè) rồi thay bằng liên kết
        for g, _, ts in os.walk(goc):
            for t in ts:
                d = os.path.join(dich, os.path.relpath(os.path.join(g, t), goc))
                if not os.path.exists(d):
                    os.makedirs(os.path.dirname(d), exist_ok=True)
                    shutil.copy2(os.path.join(g, t), d)
        shutil.rmtree(goc)
    os.makedirs(os.path.dirname(goc), exist_ok=True)
    os.symlink(dich, goc)
print("Lưu trên Drive:", ", ".join(LIEN_KET))

In [ ]:
# ===== 4. Giải nén gói dữ liệu có sẵn (nếu đã tải TN10_du_lieu.zip lên Drive) =====
import zipfile
GOI, DAU = f"{DRIVE}/TN10_du_lieu.zip", f"{DRIVE}/data/bao_da_tuyen/.da_giai_nen_TN10"
if os.path.exists(GOI) and not os.path.exists(DAU):
    with zipfile.ZipFile(GOI) as z:
        ten = [n for n in z.namelist() if not os.path.exists(os.path.join(MA, n))]
        for i, n in enumerate(ten):
            z.extract(n, MA)
            if i % 200 == 0:
                print(f"  {i}/{len(ten)}")
    open(DAU, "w").write("ok")
    print("Đã giải nén", len(ten), "tệp")
else:
    print("Không có gói hoặc đã giải nén trước đó. Thiếu tập nào thì pipeline tự sinh.")

In [ ]:
# ===== 5. Tài nguyên → tham số chạy =====
import psutil
import torch
CPU = os.cpu_count()
RAM = psutil.virtual_memory().total / 2**30
GPU = torch.cuda.is_available()
TT = max(1, min(6, CPU, int(RAM // 3)))         # tiến trình sinh / chấm (~2,5 GB RAM mỗi tiến trình)
SS = 2 if (GPU or CPU >= 4) else 1              # số mô hình huấn luyện cùng lúc
LU = max(1, CPU // SS)                          # luồng PyTorch mỗi mô hình
print(f"{CPU} CPU, {RAM:.0f} GB RAM, GPU: {torch.cuda.get_device_name(0) if GPU else 'không'}")
print(f"→ --tien-trinh {TT} --song-song {SS} --luong-cpu {LU} --thiet-bi auto")
if not GPU:
    print("⚠ CHƯA CÓ GPU: Runtime → Change runtime type → T4 GPU, rồi Run all lại. Trên CPU, huấn luyện"
          " (16 mô hình bước 1, có nhánh 120 epoch) chậm gấp nhiều lần. Hết hạn mức GPU thì vẫn chạy được trên CPU.")
if CPU <= 2:
    print("Lưu ý: chỉ 2 CPU — sinh dữ liệu và chấm sẽ chậm, cần khoảng 2–3 phiên. Mỗi phiên chỉ cần Run all lại.")

In [ ]:
# ===== 6. Kiểm tra nhanh (3–5 phút); lỗi thì DỪNG Run all, không chạy tiếp ô 7 =====
r = subprocess.run([sys.executable, "-W", "ignore", "-m", "scripts.thu_nhanh_tn10"], cwd=MA, capture_output=True, text=True)
print(r.stdout, r.stderr[-3000:])
if r.returncode:
    raise SystemExit("Kiểm tra nhanh LỖI (xem trên). Thường do push thiếu tệp: push đủ rồi Run all lại.")

In [ ]:
# ===== 7. CHẠY TN10 (dài; ngắt giữa chừng thì Run all lại, bước đã xong tự bỏ qua) =====
!cd {MA} && python -W ignore -m scripts.chay_tn10 --tien-trinh {TT} --song-song {SS} --luong-cpu {LU} --thiet-bi auto 2>&1 | tee -a logs/tn10_colab.log

In [ ]:
# ===== 8. Tiến độ: các bước đã chọn =====
import json
p = f"{MA}/results/bao_da_tuyen/tn10/chon.json"
c = json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}
for k, v in c.items():
    tom = {x: v[x] for x in ("ho", "S", "gan_nhan_lai", "trung_DM6r") if x in v}
    print(f"{k:7s} → {tom}")
if not c:
    print("Chưa có bước nào xong.")

In [ ]:
# ===== 9. Lưu kết quả lên Drive + gói để gửi lại =====
import glob
KQ = f"{MA}/results/bao_da_tuyen"
if not os.path.exists(f"{KQ}/KET_QUA_10.md") and os.path.exists(f"{KQ}/tn10/chon.json"):
    subprocess.run([sys.executable, "-W", "ignore", "-m", "scripts.phan_tich_10"], cwd=MA)
tep = [f for f in (f"{KQ}/KET_QUA_10.md", f"{KQ}/ket_qua_10.json", f"{KQ}/hinh_10.png") if os.path.exists(f)]
for f in tep:
    shutil.copy2(f, DRIVE)
gui = tep + glob.glob(f"{KQ}/tn10/**/*", recursive=True) + glob.glob(f"{KQ}/mo_hinh10/*_nhat_ky.json") \
    + glob.glob(f"{MA}/logs/tn10*.log")
with zipfile.ZipFile(f"{DRIVE}/TN10_ket_qua_colab.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in gui:
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, MA))
print("Đã lưu:", [os.path.basename(f) for f in tep], "+", f"{DRIVE}/TN10_ket_qua_colab.zip")

In [ ]:
# ===== 10. (Tuỳ chọn) đẩy kết quả lên GitHub, nhánh riêng tn10-colab =====
if DAY_KET_QUA_LEN_GITHUB:
    if not TOKEN:
        raise SystemExit("Cần GITHUB_TOKEN có quyền ghi trong Colab Secrets")
    TAM, NHANH_KQ = "/content/day_ket_qua", "tn10-colab"
    shutil.rmtree(TAM, ignore_errors=True)
    git("clone", "--branch", NHANH, "--single-branch", URL, TAM)
    git("checkout", "-B", NHANH_KQ, cwd=TAM)
    with zipfile.ZipFile(f"{DRIVE}/TN10_ket_qua_colab.zip") as z:
        z.extractall(TAM)
    git("config", "user.name", "TN10 Colab", cwd=TAM)
    git("config", "user.email", "tn10-colab@users.noreply.github.com", cwd=TAM)
    git("add", "-f", "results/bao_da_tuyen", "logs", cwd=TAM)
    git("commit", "-m", "TN10: kết quả chạy trên Colab", cwd=TAM)
    print(git("push", "-u", "origin", NHANH_KQ, cwd=TAM))
else:
    print("Bỏ qua (DAY_KET_QUA_LEN_GITHUB = False). Kết quả nằm ở", DRIVE)